<a href="https://colab.research.google.com/github/siddhantbisht2004/CSE-chatbot/blob/main/CSE_Chatbot_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSE Document Chatbot - Google Colab Edition

This notebook runs a FastAPI-based document question-answer chatbot in Google Colab.

**Features:**
- 📄 Upload PDF, DOCX, TXT documents
- 🔍 Semantic search using sentence transformers
- 💬 Answer questions based on your documents
- 🚀 FastAPI web server with interactive Swagger UI
- 🔗 Publicly accessible via ngrok tunnel

## Step 1: Install Dependencies

In [ ]:
!pip install -q PyPDF2 python-docx sentence-transformers scikit-learn fastapi uvicorn python-multipart pyngrok

## Step 2: Define Chatbot Core Logic

In [ ]:
import os
import json
import numpy as np
from typing import List, Dict
import PyPDF2
import docx
import logging
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DocumentProcessor:
    """Handles document loading and text extraction from various file formats."""

    @staticmethod
    def read_text_file(file_path: str) -> str:
        """Read content from text files."""
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()

    @staticmethod
    def read_pdf_file(file_path: str) -> str:
        """Read content from PDF files."""
        text = ""
        try:
            with open(file_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                for page in pdf_reader.pages:
                    text += page.extract_text() + "\n"
        except Exception as e:
            logger.error(f"Error reading PDF: {e}")
        return text

    @staticmethod
    def read_word_file(file_path: str) -> str:
        """Read content from Word documents."""
        try:
            doc = docx.Document(file_path)
            return "\n".join([paragraph.text for paragraph in doc.paragraphs])
        except Exception as e:
            logger.error(f"Error reading DOCX: {e}")
            return ""

    @staticmethod
    def process_directory(directory_path: str) -> List[Dict]:
        """Process all supported documents in a directory."""
        processed_docs = []
        supported_extensions = {'.txt', '.pdf', '.docx'}

        if not os.path.exists(directory_path):
            logger.warning(f"Directory {directory_path} does not exist")
            return processed_docs

        for root, _, files in os.walk(directory_path):
            for file in files:
                file_path = os.path.join(root, file)
                extension = os.path.splitext(file)[1].lower()

                if extension not in supported_extensions:
                    continue

                try:
                    print(f"Processing: {file}")
                    if extension == '.txt':
                        content = DocumentProcessor.read_text_file(file_path)
                    elif extension == '.pdf':
                        content = DocumentProcessor.read_pdf_file(file_path)
                    elif extension == '.docx':
                        content = DocumentProcessor.read_word_file(file_path)

                    chunks = DocumentProcessor.chunk_text(content)

                    for chunk in chunks:
                        processed_docs.append({
                            "content": chunk,
                            "metadata": {
                                "source": file_path,
                                "type": extension[1:],
                                "chunk_size": len(chunk)
                            }
                        })

                except Exception as e:
                    logger.error(f"Error processing {file_path}: {str(e)}")

        return processed_docs

    @staticmethod
    def chunk_text(text: str, chunk_size: int = 1000, overlap: int = 100) -> List[str]:
        """Split text into overlapping chunks."""
        chunks = []
        start = 0
        text_length = len(text)

        while start < text_length:
            end = start + chunk_size

            if end < text_length:
                while end > start and text[end] != ' ':
                    end -= 1

            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)

            start = end - overlap

        return chunks


class CSEChatbot:
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize the chatbot with necessary components."""
        print("Loading model...")
        self.encoder = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None
        print("Model loaded successfully!")

    def load_documents(self, directory_path: str):
        """Load and process documents from the specified directory."""
        logger.info(f"Processing documents from {directory_path}")

        docs = DocumentProcessor.process_directory(directory_path)
        self.documents = docs

        print(f"\n✅ Documents processed: {len(self.documents)}")
        if len(self.documents) > 0:
            print(f"Sample content (first 150 chars): {self.documents[0]['content'][:150]}...")

        if not self.documents:
            print("⚠️  No documents found. Upload documents first!")
            return

        print("Generating embeddings...")
        texts = [doc["content"] for doc in self.documents]
        self.embeddings = self.encoder.encode(texts, show_progress_bar=True)
        print(f"✅ Embeddings generated for {len(self.embeddings)} chunks")

    def save_knowledge_base(self, file_path: str):
        """Save the processed documents and embeddings."""
        data = {
            "documents": self.documents,
            "embeddings": self.embeddings.tolist() if self.embeddings is not None else None
        }

        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f)
        print(f"✅ Knowledge base saved to {file_path}")

    def load_knowledge_base(self, file_path: str):
        """Load previously processed documents and embeddings."""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if "documents" in data and "embeddings" in data:
                self.documents = data["documents"]
                self.embeddings = np.array(data["embeddings"]) if data["embeddings"] else None
                print(f"✅ Knowledge base loaded: {len(self.documents)} documents")
            else:
                print("❌ Invalid knowledge base format")
        except FileNotFoundError:
            print(f"❌ File {file_path} not found")
        except json.JSONDecodeError as e:
            print(f"❌ JSON error: {e}")

    def get_response(self, query: str, top_k: int = 3) -> Dict:
        """Process query and return response with relevant context."""
        start_time = time.time()

        query_embedding = self.encoder.encode(query)

        if self.embeddings is None or len(self.embeddings) == 0:
            return {
                "query": query,
                "response": "No documents loaded. Please upload documents first.",
                "relevant_documents": [],
                "processing_time": time.time() - start_time
            }

        similarities = cosine_similarity([query_embedding], self.embeddings)[0]

        if similarities.max() < 0.2:
            return {
                "query": query,
                "response": "No relevant information found in documents.",
                "relevant_documents": [],
                "processing_time": time.time() - start_time
            }

        top_indices = np.argsort(similarities)[-top_k:][::-1]

        relevant_docs = []
        for idx in top_indices:
            if idx < len(self.documents):
                doc = self.documents[idx]
                relevant_docs.append({
                    "content": doc["content"][:500],  # Limit to 500 chars for display
                    "metadata": doc["metadata"],
                    "similarity": float(similarities[idx])
                })

        response = self._generate_simple_response(query, relevant_docs)
        processing_time = time.time() - start_time

        return {
            "query": query,
            "response": response,
            "relevant_documents": relevant_docs,
            "processing_time": processing_time
        }

    def _generate_simple_response(self, query: str, relevant_docs: List[Dict]) -> str:
        """Generate a simple response based on the most relevant document."""
        if not relevant_docs:
            return "No relevant information found."
        return relevant_docs[0]["content"][:500]

print("✅ Chatbot core loaded!")

## Step 3: Create FastAPI Server

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import shutil
import asyncio
from pathlib import Path

# Create upload directory
upload_dir = Path("/content/uploaded_files")
upload_dir.mkdir(exist_ok=True)

# Initialize FastAPI app
app = FastAPI(
    title="CSE Document Chatbot",
    description="AI-powered document Q&A chatbot",
    version="1.0.0"
)

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Initialize chatbot
chatbot = CSEChatbot()

# Request model
class QueryRequest(BaseModel):
    query: str
    top_k: int = 3

# API Routes
@app.get("/")
async def root():
    return {
        "message": "CSE Chatbot API",
        "endpoints": {
            "docs": "/docs",
            "health": "/health",
            "status": "/status",
            "upload": "/upload",
            "query": "/query"
        }
    }

@app.get("/health")
async def health_check():
    return {"status": "healthy"}

@app.get("/status")
async def get_status():
    return {
        "loaded": chatbot.embeddings is not None,
        "documents_count": len(chatbot.documents),
        "model": "all-MiniLM-L6-v2"
    }

@app.post("/upload")
async def upload_files(files: list[UploadFile] = File(...)):
    try:
        uploaded_files = []
        for file in files:
            file_path = upload_dir / file.filename
            with open(file_path, "wb") as buffer:
                shutil.copyfileobj(file.file, buffer)
            uploaded_files.append(file.filename)
        
        # Process documents
        chatbot.load_documents(str(upload_dir))
        chatbot.save_knowledge_base("/content/knowledge_base.json")
        
        return {
            "message": "Files uploaded and processed successfully",
            "files": uploaded_files,
            "documents_loaded": len(chatbot.documents)
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/query")
async def query_chatbot(request: QueryRequest):
    try:
        result = chatbot.get_response(request.query, request.top_k)
        return result
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.delete("/knowledge-base")
async def clear_knowledge_base():
    chatbot.documents = []
    chatbot.embeddings = None
    return {"message": "Knowledge base cleared"}

@app.post("/reload-knowledge-base")
async def reload_knowledge_base():
    try:
        chatbot.load_knowledge_base("/content/knowledge_base.json")
        return {
            "message": "Knowledge base reloaded",
            "documents_loaded": len(chatbot.documents)
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

print("✅ FastAPI app configured!")

## Step 4: Setup ngrok Tunnel for Public Access

In [ ]:
# Get ngrok auth token from: https://dashboard.ngrok.com/auth
# IMPORTANT: Replace with your actual ngrok token!

ngrok_token = "your_ngrok_token_here"  # <-- REPLACE THIS

if ngrok_token == "your_ngrok_token_here":
    print("\n⚠️  NGROK TOKEN NOT SET")
    print("\nTo get public URL, follow these steps:")
    print("1. Go to https://dashboard.ngrok.com/auth")
    print("2. Copy your auth token")
    print("3. Replace 'your_ngrok_token_here' in the cell above")
    print("4. Run this cell again")
    print("\nWithout ngrok, you can still access:")
    print("- Local: http://localhost:8000/docs")
else:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    print("✅ ngrok configured with your token")

## Step 5: Start the Server

In [ ]:
import uvicorn
from pyngrok import ngrok
import threading
import time

# Get ngrok public URL if token is set
ngrok_url = None
try:
    if ngrok_token != "your_ngrok_token_here":
        # Create ngrok tunnel
        ngrok_tunnel = ngrok.connect(8000, "http")
        ngrok_url = ngrok_tunnel.public_url
        print(f"\n🌐 Public URL: {ngrok_url}")
        print(f"🌐 Public Docs: {ngrok_url}/docs")
except:
    print("\nℹ️  Running in local mode only")

# Start server in a thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

print("\n" + "="*60)
print("✅ SERVER STARTED SUCCESSFULLY!")
print("="*60)
print("\n📍 Access the chatbot:")
if ngrok_url:
    print(f"   Public:  {ngrok_url}/docs")
else:
    print("   Local:   http://localhost:8000/docs")
print("\n📝 Next Steps:")
print("   1. Run the next cell to upload documents")
print("   2. Then run the query example cell")
print("\n" + "="*60)

## Step 6: Upload Documents

In [ ]:
from google.colab import files

print("📤 Upload your documents (PDF, DOCX, TXT)")
print("Click 'Choose Files' below...\n")

uploaded = files.upload()

for filename in uploaded.keys():
    file_path = upload_dir / filename
    with open(file_path, 'wb') as f:
        f.write(uploaded[filename])
    print(f"✅ Uploaded: {filename}")

print("\n📊 Processing documents...")
chatbot.load_documents(str(upload_dir))
chatbot.save_knowledge_base("/content/knowledge_base.json")

print(f"\n✅ Ready! You have {len(chatbot.documents)} document chunks")

## Step 7: Query the Chatbot

In [ ]:
import requests
import json

# Example queries
queries = [
    "What is CSE?",
    "Tell me about the curriculum",
    "What are the admission requirements?"
]

# Query the chatbot
query = queries[0]  # Change index to try different queries
print(f"\n❓ Query: {query}\n")

try:
    response = requests.post(
        "http://localhost:8000/query",
        json={"query": query, "top_k": 3}
    )
    
    result = response.json()
    
    print("📝 Response:")
    print("-" * 60)
    print(result["response"][:500])  # Show first 500 chars
    print("-" * 60)
    
    print(f"\n📊 Found {len(result['relevant_documents'])} relevant documents")
    
    for i, doc in enumerate(result['relevant_documents'], 1):
        print(f"\n{i}. Similarity: {doc['similarity']:.2%}")
        print(f"   Source: {doc['metadata']['source']}")
    
    print(f"\n⏱️  Processing time: {result['processing_time']:.3f}s")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("Make sure to upload documents first!")

## Step 8: Interactive Query Interface

In [ ]:
import requests

def ask_chatbot(question, top_k=3):
    """Ask the chatbot a question and get response with sources."""
    try:
        response = requests.post(
            "http://localhost:8000/query",
            json={"query": question, "top_k": top_k},
            timeout=30
        )
        
        result = response.json()
        
        print("\n" + "="*60)
        print(f"Q: {question}")
        print("="*60)
        print(f"\nA: {result['response'][:500]}...")
        print("\n" + "-"*60)
        print(f"📊 Similarity Scores:")
        for i, doc in enumerate(result['relevant_documents'], 1):
            print(f"  {i}. {doc['similarity']:.1%} - {doc['metadata']['source'].split('/')[-1]}")
        print(f"\n⏱️  Time: {result['processing_time']:.3f}s")
        print("="*60 + "\n")
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Ensure documents are uploaded and server is running.")

# Example usage
ask_chatbot("What is the Computer Science department?")

## Step 9: Check Server Status

In [ ]:
import requests
import json

try:
    # Check health
    health = requests.get("http://localhost:8000/health").json()
    print(f"🏥 Health: {health['status']}")
    
    # Check status
    status = requests.get("http://localhost:8000/status").json()
    print(f"\n📊 Chatbot Status:")
    print(f"  - Loaded: {status['loaded']}")
    print(f"  - Documents: {status['documents_count']} chunks")
    print(f"  - Model: {status['model']}")
    
    print("\n✅ Server is running correctly!")
    
except Exception as e:
    print(f"❌ Server error: {e}")
    print("Make sure the server is running (Step 5)")

## 💡 Tips & Troubleshooting

### Uploading Documents
- Supported formats: PDF, DOCX, TXT
- Works best with structured documents
- Maximum file size depends on Colab limits

### Query Tips
- Ask clear, specific questions
- The chatbot finds similar content chunks
- Similarity score shows match quality (0-1)

### Using Public URL (ngrok)
1. Get free ngrok account: https://dashboard.ngrok.com/
2. Copy your auth token
3. Paste in Step 4 cell
4. Access from anywhere via public URL

### If Something Goes Wrong
1. Check if server is running (Step 5)
2. Verify documents uploaded (Step 6)
3. Try Step 9 to check status
4. Restart the notebook and run again

### Performance
- First query may be slow (model warm-up)
- Subsequent queries are faster
- Processing time shown in results

## 🚀 Advanced Usage

In [ ]:
# Batch Query Multiple Questions
import requests
import pandas as pd

questions = [
    "What is CSE?",
    "What are the courses offered?",
    "How to apply?",
]

results = []
for q in questions:
    try:
        response = requests.post(
            "http://localhost:8000/query",
            json={"query": q, "top_k": 1}
        )
        result = response.json()
        results.append({
            "Question": q,
            "Answer": result["response"][:100],
            "Similarity": result["relevant_documents"][0]["similarity"] if result["relevant_documents"] else 0,
            "Time (s)": result["processing_time"]
        })
    except:
        pass

# Display results as table
df = pd.DataFrame(results)
print("\n📋 Batch Query Results:\n")
print(df.to_string(index=False))